<a href="https://colab.research.google.com/github/2303a51355/High-performance-computing/blob/main/Lab_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1.Identification of NUMA Architecture and System Topology

In [1]:
!lscpu | grep NUMA

NUMA node(s):                            1
NUMA node0 CPU(s):                       0,1


In [4]:
!apt-get update
!apt-get install -y numactl

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.0 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,769 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,538 kB]
Ge

In [6]:
!numactl --hardware

available: 1 nodes (0)
node 0 cpus: 0 1
node 0 size: 12975 MB
node 0 free: 8641 MB
node distances:
node   0 
  0:  10 


In [8]:
!lscpu

Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             46 bits physical, 48 bits virtual
  Byte Order:                Little Endian
CPU(s):                      2
  On-line CPU(s) list:       0,1
Vendor ID:                   GenuineIntel
  Model name:                Intel(R) Xeon(R) CPU @ 2.20GHz
    CPU family:              6
    Model:                   79
    Thread(s) per core:      2
    Core(s) per socket:      1
    Socket(s):               1
    Stepping:                0
    BogoMIPS:                4400.36
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush mmx fxsr sse sse2 ss h
                             t syscall nx pdpe1gb rdtscp lm constant_tsc rep_goo
                             d nopl xtopology nonstop_tsc cpuid tsc_known_freq p
                             ni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2ap
                   

2.Memory Bandwidth Measurement on Single NUMA Node

In [12]:
import numpy as np
import time
size = 100_000_000
a = np.ones(size)
b = np.ones(size)
start = time.time()
for i in range(5):
    c = a + b
    d = c * 2
end = time.time()
total_bytes = a.nbytes * 2 * 5
time_taken = end - start
bandwidth = total_bytes / time_taken / (1024**3)
print("Execution Time:", time_taken, "seconds")
print("Effective Bandwidth:", bandwidth, "GB/s")

Execution Time: 4.1768693923950195 seconds
Effective Bandwidth: 1.7837715037222317 GB/s


3.Local vs Remote Memory Access Performance (NUMA-Aware Execution)


In [10]:
!numactl --hardware

available: 1 nodes (0)
node 0 cpus: 0 1
node 0 size: 12975 MB
node 0 free: 8977 MB
node distances:
node   0 
  0:  10 


In [11]:
import numpy as np
import time
size_mb = 500
num_elements = (size_mb * 1024 * 1024) // 8
arr = np.ones(num_elements, dtype=np.float64)
start = time.time()
for _ in range(5):
    arr = arr * 2.0
end = time.time()
elapsed = end - start
bandwidth = (size_mb * 5) / elapsed
print("Execution Time:", elapsed, "seconds")
print("Effective Bandwidth:", bandwidth, "MB/s")

Execution Time: 0.8393633365631104 seconds
Effective Bandwidth: 2978.4479391685095 MB/s


4.Impact of Multi-threaded Memory Access on NUMA Systems4

In [13]:
import numpy as np
import threading
import time

SIZE_MB = 400
NUM_THREADS = 4

num_elements = (SIZE_MB * 1024 * 1024) // 8
arr = np.ones(num_elements, dtype=np.float64)
def worker():
    global arr
    for _ in range(5):
        arr = arr * 1.1
threads = []
start = time.time()
for i in range(NUM_THREADS):
    t = threading.Thread(target=worker)
    threads.append(t)
    t.start()
for t in threads:
    t.join()
end = time.time()
elapsed = end - start
bandwidth = (SIZE_MB * 5 * NUM_THREADS) / elapsed
print("Execution Time:", elapsed, "seconds")
print("Effective Bandwidth:", bandwidth, "MB/s")

Execution Time: 2.741024971008301 seconds
Effective Bandwidth: 2918.6162419589914 MB/s


5.Comparative Study of NUMA-Unaware vs NUMA-Aware Execution

In [14]:
import numpy as np
import time

SIZE_MB = 500
num_elements = (SIZE_MB * 1024 * 1024) // 8

arr = np.ones(num_elements, dtype=np.float64)

start = time.time()
for _ in range(6):
    arr = arr * 1.5
end = time.time()

elapsed = end - start
bandwidth = (SIZE_MB * 6) / elapsed

print("Execution Time:", elapsed, "seconds")
print("Effective Bandwidth:", bandwidth, "MB/s")

Execution Time: 1.0196425914764404 seconds
Effective Bandwidth: 2942.207421578973 MB/s
